# Full car: the four events

Runs the acceleration, skidpad, autocross and endurance for either car with the quasi-steady solver,
after you change any value of its car file (`cars/<car>/vehicle.yaml`). The sliders cover the values that
move lap time most; for any other value add a line to `EXTRA`, using the same keys the website exports
(`vehicle:...`, `pt:...`, `ev:...`, `pack:...`).

Model: `lapsim/solvers/qss.py` (method), `qss_ic.py` / `qss_ev.py` (the power modules' solvers).

In [ ]:
%matplotlib inline
from nb import *

In [ ]:
EXTRA = {}          # e.g. {"vehicle:front.spring_rate": "300 lbf/in", "ev:strategy.power_limit": "60 kW"}

car_pick = W.ToggleButtons(options=list(CARS), description="Car")
events = W.SelectMultiple(options=lapsim.core.events.EVENTS, value=("acceleration", "skidpad", "autocross"),
                          description="Events", rows=4)
out = W.Output()
box = W.VBox()


def sliders_for(car):
    return [slider(car, "vehicle:mass.mass_no_driver", 150, 300, 1, "mass without driver", 1, "kg"),
            slider(car, "vehicle:mass.cg_height", 200, 350, 1, "CG height", 1e3, "mm"),
            slider(car, "vehicle:mass.front_weight_fraction", 0.40, 0.60, 0.005, "front weight fraction"),
            slider(car, "vehicle:tires.lambda_mu", 0.8, 1.2, 0.01, "grip scale (lambda_mu)"),
            slider(car, "vehicle:aero.calib_ClA", 0.0, 4.0, 0.05, "ClA", 1, "m^2"),
            slider(car, "vehicle:aero.calib_CdA", 0.5, 2.0, 0.02, "CdA", 1, "m^2")]


def build(_=None):
    global CAR, SL
    CAR = lapsim.load_car(CARS[car_pick.value], EXTRA)
    SL = sliders_for(CAR)
    box.children = [*SL, run_button("Run events", run, out)]


def run():
    c = with_values(CAR, SL)
    res = c.run(events.value)
    display(res.table().round(3))
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    for e in events.value:
        if e in res.laps and e != "skidpad":
            lr = res.laps[e]
            ax[0].plot(lr.s, 3.6 * lr.v, label=e)
    ax[0].set(xlabel="distance [m]", ylabel="speed [km/h]", title="Speed along the lap"); ax[0].legend()
    lr = res.laps.get("autocross") or res.laps.get(events.value[0])
    lim = pd.Series(lr.limit).value_counts(normalize=True) * 100
    lim.plot.barh(ax=ax[1]); ax[1].set(xlabel="% of the distance", title=f"What limits the car ({lr.track.name})")
    plt.show()
    for n in res.notes:
        print("note:", n)


car_pick.observe(build, "value")
build()
display(W.VBox([car_pick, events, box, out]))
run_button("", run, out).click()